In [11]:
import os
import ast
import time
import pandas as pd
import requests
from dotenv import load_dotenv

# Load API key
load_dotenv()
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    raise ValueError("API Key not found! Please check your .env file.")

# Set model to GPT-5.6 Luna
MODEL = "openai/gpt-5.6-luna"
print(f"Libraries loaded. Target Model set to: {MODEL}")

Libraries loaded. Target Model set to: openai/gpt-5.6-luna


In [12]:
# Adjust the input path if needed
INPUT_FILE = "../../../dataset/NER/test_tag_columns.csv"  
OUTPUT_FILE = "gpt5.6_luna_health_condition_full_dataset.csv"

def extract_first_entity(val):
    """Safely extracts the first valid entity from various column formats."""
    if pd.isna(val): return None
    if isinstance(val, list):
        return str(val[0]).strip() if len(val) > 0 and str(val[0]).strip() else None

    val_str = str(val).strip()
    if not val_str or val_str in ["[]", "nan", "None", ""]: return None

    if val_str.startswith("[") and val_str.endswith("]"):
        try:
            parsed = ast.literal_eval(val_str)
            if isinstance(parsed, list) and len(parsed) > 0:
                first_elem = str(parsed[0]).strip()
                return first_elem if first_elem else None
        except Exception:
            pass
    return val_str

print(f"Loading input file: {INPUT_FILE}...")
df_raw = pd.read_csv(INPUT_FILE)

# Extract and clean the entity
df_raw["target_health_condition"] = df_raw["Health Condition"].apply(extract_first_entity)

# Filter for rows that actually have a Health Condition
df_valid = df_raw.dropna(subset=["target_health_condition"]).copy()
df_valid = df_valid[df_valid["target_health_condition"] != ""].reset_index(drop=True)

# Use the entire valid dataset (No SAMPLE_SIZE limit)
df_full = df_valid.copy()

print(f"Total valid Health Condition rows found: {len(df_full)}")
print(f"Ready to process ALL {len(df_full)} rows.")

Loading input file: ../../../dataset/NER/test_tag_columns.csv...
Total valid Health Condition rows found: 577
Ready to process ALL 577 rows.


In [13]:
PROMPT_TEMPLATE = """You are given a Bangla medical sentence containing one or more health condition entities.

Your task is to create a modified version of the sentence by replacing exactly ONE health condition entity with a different but closely related health condition.

Rules:
1. Identify the specified health condition entity in the sentence.
2. Replace exactly ONE occurrence of that health condition with another health condition.
3. The replacement must be different from the original health condition.
4. The replacement must NOT be another health condition already present in the original sentence.
5. The replacement should be medically plausible and closely related to the original condition in terms of disease type, body system, pathology, or clinical context.
6. Prefer a condition that could reasonably be confused with, resemble, coexist with, or be clinically related to the original condition.
7. Do NOT replace the health condition with an unrelated disease simply because it is common.
8. The replacement must fit naturally into the surrounding sentence without making the sentence medically or linguistically implausible.
9. Do not add any additional health condition.
10. Do not remove, add, or modify any other information in the sentence.
11. Keep the rest of the sentence exactly as unchanged as possible.
12. Do not modify the original NER annotation.
13. Do not provide explanations or identify the replacement in prose.
14. Return your answer in EXACTLY the following format:

Modified Health Condition: <new health condition>

Modified Sentence: <modified Bangla sentence>

Example:

Original sentence:
রোগীর ডায়াবেটিস রয়েছে এবং তিনি নিয়মিত ওষুধ গ্রহণ করেন।

Health condition entity:
ডায়াবেটিস

A suitable replacement should be a medically related health condition, rather than an arbitrary or unrelated disease.

Output:

Modified Health Condition: উচ্চ রক্তচাপ

Modified Sentence: রোগীর উচ্চ রক্তচাপ রয়েছে এবং তিনি নিয়মিত ওষুধ গ্রহণ করেন।

Now perform the replacement.

Original sentence:
{sentence}

Health condition entity:
{health_condition}
"""

def replace_health_condition(sentence, health_condition):
    prompt = PROMPT_TEMPLATE.format(
        sentence=str(sentence).strip(), 
        health_condition=str(health_condition).strip()
    )

    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.0 # Kept at 0.0 as per your team's strict output requirements
    }

    response = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers=headers,
        json=payload,
        timeout=120
    )
    response.raise_for_status()

    content = response.json()["choices"][0]["message"]["content"].strip()

    modified_health_condition = ""
    modified_sentence = ""

    # Parse the exact format requested in Rule 14
    for line in content.splitlines():
        if line.startswith("Modified Health Condition:"):
            modified_health_condition = line.replace("Modified Health Condition:", "").strip()
        elif line.startswith("Modified Sentence:"):
            modified_sentence = line.replace("Modified Sentence:", "").strip()

    return modified_health_condition, modified_sentence

In [14]:
modified_health_conditions = []
modified_sentences = []

print(f"Starting FULL DATASET execution loop on {len(df_full)} rows using {MODEL}...\n")

for idx, row in df_full.iterrows():
    sentence = row["text"]
    health_condition = row["target_health_condition"]

    print(f"[{idx+1}/{len(df_full)}] Swapping: '{health_condition}'...")

    try:
        new_condition, new_sentence = replace_health_condition(sentence, health_condition)
        print(f"   -> Result: '{new_condition}'")
    except Exception as e:
        print(f"   -> Error: {e}")
        new_condition = ""
        new_sentence = ""

    modified_health_conditions.append(new_condition)
    modified_sentences.append(new_sentence)

    time.sleep(0.5)  # Rate limit cooldown

    # AUTO-SAVE LOGIC: Save a backup every 50 rows just in case of failure
    if (idx + 1) % 50 == 0:
        df_full.loc[:idx, "modified_health_condition"] = modified_health_conditions
        df_full.loc[:idx, "modified_sentence"] = modified_sentences
        df_full.to_csv("backup_" + OUTPUT_FILE, index=False, encoding="utf-8-sig")
        print(f"   --- Auto-saved backup at row {idx+1} ---")

# Final Save
df_full["modified_health_condition"] = modified_health_conditions
df_full["modified_sentence"] = modified_sentences

export_cols = [
    "text",
    "Health Condition",
    "target_health_condition",
    "modified_health_condition",
    "modified_sentence",
]
available_export_cols = [c for c in export_cols if c in df_full.columns]

df_full[available_export_cols].to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
print(f"\nExecution complete! Final dataset saved to: {OUTPUT_FILE}")

Starting FULL DATASET execution loop on 577 rows using openai/gpt-5.6-luna...

[1/577] Swapping: 'হার্ট এটাক | স্ট্রোকের'...
   -> Result: 'হৃদরোগ'
[2/577] Swapping: 'দুশ্চিন্তা বা এনজাইটি ডিসঅর্ডার'...
   -> Result: 'এসেনশিয়াল ট্রেমর'
[3/577] Swapping: 'heart disease'...
   -> Result: 'cardiovascular disease'
[4/577] Swapping: 'diabetes | pressure | gas | thyroid problems'...
   -> Result: 'GERD'
[5/577] Swapping: 'pregnancy'...
   -> Result: 'ectopic pregnancy'
[6/577] Swapping: 'বিড়াল খুব সামান্য আচর দিয়েছে'...
   -> Result: 'বিড়াল খুব সামান্য কামড় দিয়েছে'
[7/577] Swapping: 'osteoporosis'...
   -> Result: 'rheumatoid arthritis'
[8/577] Swapping: '১৪ দিনের প্রেগন্যান্ট | মাসিক'...
   -> Result: 'গর্ভপাত'
[9/577] Swapping: 'ইনফেকশন'...
   -> Result: 'প্রদাহ'
[10/577] Swapping: 'ডিপ্রেশনে | উচ্চ রক্তচাপের'...
   -> Result: 'উদ্বেগ'
[11/577] Swapping: 'পিরিয়ড'...
   -> Result: 'মাসিক'
[12/577] Swapping: 'hypertension'...
   -> Result: 'hypotension'
[13/577] Swapping: 'থাইরাইর সমস্যা'..